In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")                     # non-interactive backend for saving
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import scipy.stats as stats
from scipy.stats import shapiro, zscore
from patsy import dmatrix
from sklearn.linear_model import LinearRegression
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.formula.api as smf
import pickle
import os

In [2]:
# Output directory — all plots and tables land here
OUT = "/content/sample_data"
os.makedirs(OUT, exist_ok=True)

# Seed for reproducibility
np.random.seed(42)

print("=" * 70)
print("  TOASTER REVIEWS — H1 & H2  (Python)")
print("=" * 70)


  TOASTER REVIEWS — H1 & H2  (Python)


In [3]:
# =============================================================================
#  1. DATA LOADING & PRE-PROCESSING
# =============================================================================
print("\n[1/8]  Loading & pre-processing data...")

df = pd.read_excel("/content/sample_data/Toaster Data_2018_23.xlsx")

# ── Parse dates ───────────────────────────────────────────────────────────────
df["RV_DT"] = pd.to_datetime(df["RV_DT"])



[1/8]  Loading & pre-processing data...


In [4]:
# ── De-duplicate: one review per (reviewer, product), most recent first ───────
# Equivalent to: arrange(RVR, ASIN, desc(RV_DT)) |> distinct(RVR, ASIN, ...)
df = (
    df.sort_values(["RVR", "ASIN", "RV_DT"], ascending=[True, True, False])
      .drop_duplicates(subset=["RVR", "ASIN"], keep="first")
      .reset_index(drop=True)
)

# ── Coerce numeric columns (handles '.' and other non-numeric tokens) ─────────
num_cols = [
    "FS", "HLP_VT", "RVS_L", "SUBJ", "SUBJ_CD",
    "NG_RVS", "NU_RVS", "PS_RVS", "CP_RVS",
    "NGE_RVS", "PSE_RV", "TTL_RV", "RSR", "VP", "IMG_PRST",
]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# ── str(df) equivalent ────────────────────────────────────────────────────────
print(f"\n  Raw shape after dedup : {df.shape}")
print(f"  Columns               : {list(df.columns)}")

# ── Filter: products with >= 3 distinct-reviewer reviews ─────────────────────
df = df.groupby("ASIN").filter(lambda x: len(x) >= 3)


  Raw shape after dedup : (62014, 39)
  Columns               : ['ASIN', 'P_TITLE', 'OP', 'DP', 'SP', 'FS', 'RA', 'F_RETURN', 'PRA_4.5', 'P_RTG', 'RTG_P_NO', 'SELLER_LINK', 'IMAGE_URL', 'P_URL', 'RV_URL', 'PRFL_IMG', 'PRFL_URL', 'RV_TTL', 'RVS', 'RVR', 'RSR', 'RVR_CONT', 'RV_DT', 'VP', 'HLP_VT', 'IMG_PRST', 'TTL_RV', 'RVS_L', 'RV_TRANS', 'SUBJ', 'SUBJ_CD', 'SRVS', 'SRVSCD', 'NG_RVS', 'NU_RVS', 'PS_RVS', 'CP_RVS', 'NGE_RVS', 'PSE_RV']


In [5]:
# ── Drop missing RSR / CP_RVS; short reviews (RVS_L <= 20) ───────────────────
df = df.dropna(subset=["RSR", "CP_RVS"])
df = df[df["RVS_L"] > 20].copy()

# ── summary(df) equivalent ───────────────────────────────────────────────────
print("\n  Summary of key numeric columns:")
print(df[["RSR", "CP_RVS", "HLP_VT", "RVS_L", "TTL_RV", "SUBJ"]].describe().round(3))

# ── Quarter variable (matches R's paste0(year, "Q", quarter)) ─────────────────
df["quarter"] = (
    df["RV_DT"].dt.year.astype(str) + "Q" +
    df["RV_DT"].dt.quarter.astype(str)
)

# ── Scale sentiment (z-score of CP_RVS) ──────────────────────────────────────
df["sentiment"] = zscore(df["CP_RVS"].astype(float), nan_policy="omit")

print(f"\n  Post-filter shape: {df.shape}")


  Summary of key numeric columns:
             RSR     CP_RVS     HLP_VT      RVS_L     TTL_RV       SUBJ
count  56051.000  56051.000  13297.000  56051.000  55991.000  56051.000
mean       3.714      0.461      4.352    199.318   2602.812      0.547
std        1.603      0.466     22.400    236.768   2083.407      0.248
min        1.000     -0.970      1.000     21.000      3.000      0.000
25%        2.000      0.077      1.000     63.000    716.000      0.425
50%        5.000      0.625      1.000    127.000   2220.000      0.583
75%        5.000      0.848      3.000    248.000   3977.000      0.708
max        5.000      1.000   1466.000   5979.000  14944.000      1.000

  Post-filter shape: (56051, 41)


In [6]:
# =============================================================================
#  2. STAGE-1 BASELINE MODEL
#     RSR ~ B-spline(sentiment, df=4)
#     Residual = distortion (rating minus sentiment-predicted rating)
# =============================================================================
print("\n[2/8]  Stage-1 spline baseline → distortion...")

# B-spline basis with 4 degrees of freedom (matches R's bs(sentiment, df=4))
spline_basis = dmatrix(
    "bs(sentiment, df=4, include_intercept=False)",
    data=df, return_type="dataframe"
)

# OLS fit: RSR ~ spline(sentiment)
baseline_lm = LinearRegression().fit(spline_basis, df["RSR"])
df["pred_rating"] = baseline_lm.predict(spline_basis)
df["distortion"]  = df["RSR"] - df["pred_rating"]

print(f"  Baseline R² = {baseline_lm.score(spline_basis, df['RSR']):.4f}")
print(f"  Distortion  mean={df['distortion'].mean():.4f}  "
      f"sd={df['distortion'].std():.4f}")


[2/8]  Stage-1 spline baseline → distortion...
  Baseline R² = 0.3234
  Distortion  mean=0.0000  sd=1.3184


In [7]:
# =============================================================================
#  3. FEATURE ENGINEERING
#     Mirrors the second mutate() block in Toaster2.R
# =============================================================================
print("\n[3/8]  Feature engineering...")

df["HLP_VT_zero"] = df["HLP_VT"].fillna(0).astype(float)
df["HLP_missing"] = df["HLP_VT"].isna().astype(int)
df["log_length"]  = np.log(df["RVS_L"].astype(float))
df["log_helpful"] = np.log(1.0 + df["HLP_VT_zero"])
df["log_reviews"] = np.log(df["TTL_RV"].astype(float))

# Factor equivalents
df["VP"]      = pd.Categorical(df["VP"].astype(str))
df["IMG"]     = pd.Categorical(df["IMG_PRST"].astype(str))
df["product"] = pd.Categorical(df["ASIN"])
df["brand"]   = pd.Categorical(df["P_TITLE"].str.split().str[0])
df["quarter"] = pd.Categorical(df["quarter"])

# ── Prior mean rating: cumulative mean of RSR up to (but not including) review t
# Equivalent to: lag(cumsum(RSR)) / lag(row_number())
df = df.sort_values(["product", "RV_DT"])

def _prior_mean(group):
    cs = group["RSR"].astype(float).cumsum().shift(1)
    cc = pd.Series(range(len(group)), index=group.index, dtype=float)
    return cs / cc

df["prior_mean_rating"] = df.groupby("product", group_keys=False).apply(_prior_mean)

# Remove first review per product (no prior info) — mirrors filter(!is.na(...))
df = df.dropna(subset=["prior_mean_rating"]).copy()

# ── Centre sentiment (scale=FALSE in R: centre only, no rescaling) ─────────────
df["sentiment_c"] = df["sentiment"] - df["sentiment"].mean()

print(f"  Final modelling N = {len(df):,}")
print(f"  Products: {df['product'].nunique()}  "
      f"Brands: {df['brand'].nunique()}  "
      f"Quarters: {df['quarter'].nunique()}")

# ── Numeric interaction terms (needed for VIF and cluster-robust model) ────────
df["VP1"]         = (df["VP"].astype(str) == "1").astype(float)
df["IMG1"]        = (df["IMG"].astype(str) == "1").astype(float)
df["sent_VP"]     = df["sentiment_c"] * df["VP1"]
df["sent_IMG"]    = df["sentiment_c"] * df["IMG1"]
df["sent_VP_IMG"] = df["sentiment_c"] * df["VP1"] * df["IMG1"]
df["VP_IMG"]      = df["VP1"] * df["IMG1"]

# ── Final clean modelling frame ────────────────────────────────────────────────
mod_cols = [
    "distortion", "sentiment_c", "VP1", "IMG1", "VP_IMG",
    "sent_VP", "sent_IMG", "sent_VP_IMG",
    "prior_mean_rating", "log_length", "log_helpful", "log_reviews", "SUBJ",
    "quarter", "product", "brand",
]
df_mod = df[mod_cols].dropna().reset_index(drop=True)
df_mod["product_str"] = df_mod["product"].astype(str)
df_mod["brand_str"]   = df_mod["brand"].astype(str)
df_mod["quarter_str"] = df_mod["quarter"].astype(str)
print(f"  After final NA drop: N = {len(df_mod):,}")




[3/8]  Feature engineering...
  Final modelling N = 55,728
  Products: 322  Brands: 95  Quarters: 21
  After final NA drop: N = 55,668


In [28]:
# =============================================================================
#  4. model_final — MIXED LINEAR MODEL  (H1 main model)
#     Equivalent to lmer(distortion ~ VP:IMG+sentiment_c*(VP+IMG)+controls +
#                        factor(quarter) + (1|product) + (1|brand), ...)
#
#     Implementation note:
#       statsmodels MixedLM supports one grouping factor natively.
#       Brand random effect is approximated by brand dummies (fixed) for the
#       main inference model, with a separate null model used for ICC.
#       Product is the primary grouping variable (random intercept).
# =============================================================================
print("\n[4/8]  Fitting model_final (MixedLM)...")

# quarter → FIXED  (C(quarter_str))
# product → RANDOM intercept  (groups=)
# brand   → RANDOM intercept  (vc_formula=)   ← corrected from previous version
formula_mlm = (
    "distortion ~ VP1 + IMG1 + VP_IMG + "
    "sent_VP + sent_IMG + sent_VP_IMG + "
    "prior_mean_rating + log_length + log_helpful + log_reviews + SUBJ + "
    "C(quarter_str)"
)

# REML=True matches R's default; powell optimizer mirrors bobyqa stability
mlm     = smf.mixedlm(
    formula_mlm, df_mod,
    groups     = df_mod["product_str"],
    vc_formula = {"brand": "0 + C(brand_str)"}   # brand as second RE
)
mlm_fit = mlm.fit(reml=True, method="powell", maxiter=500)




[4/8]  Fitting model_final (MixedLM)...


In [29]:
# ── summary(model_final) ────────────────

print("\n  ── Fixed Effects (MixedLM) ──")
fe_table   = mlm_fit.summary().tables[1]
fe_display = fe_table[fe_table.index != "brand Var"]
print(fe_display.to_string())
# Store fitted values and residuals for diagnostics
df_mod["fitted"]  = mlm_fit.fittedvalues
df_mod["resid"]   = mlm_fit.resid

# Product random effects: recovered from the null product model above
df_mod["re_prod"] = df_mod["product_str"].map(
    {k: float(v["Group"]) for k, v in _null_prod.random_effects.items()}
)



  ── Fixed Effects (MixedLM) ──
                           Coef. Std.Err.        z  P>|z|  [0.025  0.975]
Intercept                  1.041    0.132    7.895  0.000   0.783   1.300
C(quarter_str)[T.2018Q2]  -0.009    0.045   -0.204  0.838  -0.098   0.079
C(quarter_str)[T.2018Q3]   0.044    0.048    0.923  0.356  -0.050   0.138
C(quarter_str)[T.2018Q4]   0.097    0.047    2.060  0.039   0.005   0.190
C(quarter_str)[T.2019Q1]   0.045    0.045    1.009  0.313  -0.042   0.132
C(quarter_str)[T.2019Q2]   0.155    0.047    3.280  0.001   0.062   0.247
C(quarter_str)[T.2019Q3]   0.087    0.045    1.930  0.054  -0.001   0.175
C(quarter_str)[T.2019Q4]   0.049    0.044    1.132  0.258  -0.036   0.135
C(quarter_str)[T.2020Q1]   0.051    0.040    1.268  0.205  -0.028   0.129
C(quarter_str)[T.2020Q2]  -0.102    0.042   -2.417  0.016  -0.185  -0.019
C(quarter_str)[T.2020Q3]  -0.009    0.040   -0.226  0.821  -0.087   0.069
C(quarter_str)[T.2020Q4]  -0.070    0.039   -1.794  0.073  -0.147   0.007
C(qua

In [30]:
# ── Variance components ───────────────────────────────────────────────────────
# When vc_formula is used, cov_re is empty; brand variance lives in vcomp[0].
# A separate product-only null model recovers Var(product) for reporting.
_null_prod   = smf.mixedlm("distortion ~ 1", df_mod,
                             groups=df_mod["product_str"]).fit(
                             reml=True, method="powell", maxiter=300)
var_product  = float(_null_prod.cov_re.iloc[0, 0])
var_brand_re = float(mlm_fit.vcomp[0])
var_resid_re = float(mlm_fit.scale)

print(f"  Converged   : {mlm_fit.converged}")
print(f"  Var(product): {var_product:.4f}  [from null model — cov_re empty when vc_formula used]")
print(f"  Var(brand)  : {var_brand_re:.4f}  [from mlm_fit.vcomp[0]]")
print(f"  Var(resid)  : {var_resid_re:.4f}")

# ── isSingular() equivalent ───────────────────────────────────────────────────
# Singular if either RE variance is effectively zero
is_singular = (var_product < 1e-4) or (var_brand_re < 1e-4)
print(f"  isSingular  : {is_singular}")

  Converged   : True
  Var(product): 0.0565  [from null model — cov_re empty when vc_formula used]
  Var(brand)  : 0.0321  [from mlm_fit.vcomp[0]]
  Var(resid)  : 1.4681
  isSingular  : False


In [31]:
# =============================================================================
#  5. ASSUMPTION DIAGNOSTICS  (mirrors R's assumption checks section)
#     Five panels: QQ residuals, Fitted vs Residuals, RE QQ product,
#                  RE QQ brand (from null model), Scale-Location
# =============================================================================
print("\n[5/8]  Running assumption diagnostics...")
# ── 5a. Brand random effects for QQ plot ─────────────────────────────────────
# Brand REs from the main model live in vcomp (scalar variance), not random_effects.
# We fit a brand-grouped null model to recover per-brand intercepts for the QQ plot.
null_brand     = smf.mixedlm("distortion ~ 1", df_mod, groups=df_mod["brand_str"])
null_brand_fit = null_brand.fit(reml=True, method="powell", maxiter=500)
re_brand       = np.array([float(v["Group"])
                            for v in null_brand_fit.random_effects.values()])

# ── Shapiro-Wilk on a random sample of residuals (max 5000) ──────────────────
resid_arr    = df_mod["resid"].values
fitted_arr   = df_mod["fitted"].values
sample_resid = resid_arr[np.random.choice(len(resid_arr),
                         size=min(5000, len(resid_arr)), replace=False)]
sw_stat, sw_p = shapiro(sample_resid)
print(f"  Shapiro-Wilk W={sw_stat:.4f}  p={sw_p:.4e}  "
      f"({'non-normal' if sw_p < 0.05 else 'normal'} at α=0.05)")



[5/8]  Running assumption diagnostics...
  Shapiro-Wilk W=0.9771  p=1.2315e-27  (non-normal at α=0.05)


In [32]:
# ── Build diagnostic figure (6 panels) ────────────────────────────────────────
fig = plt.figure(figsize=(16, 14))
fig.suptitle(
    "model_final — Assumption Diagnostics (H1/H2)",
    fontsize=15, fontweight="bold", y=0.98
)
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.42, wspace=0.32)

# ── Panel 1: QQ plot of residuals ─────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
(osm, osr), (slope, intercept, r) = stats.probplot(resid_arr, dist="norm")
ax1.scatter(osm, osr, alpha=0.25, s=4, color="#4472C4", rasterized=True)
x_line = np.array([osm.min(), osm.max()])
ax1.plot(x_line, slope * x_line + intercept, "r-", lw=1.5)
ax1.set_title("QQ Plot — Residuals\n"
              f"(Shapiro-Wilk p={sw_p:.2e})", fontsize=11)
ax1.set_xlabel("Theoretical Quantiles")
ax1.set_ylabel("Sample Quantiles")
ax1.annotate(f"W = {sw_stat:.4f}\np = {sw_p:.2e}",
             xy=(0.05, 0.88), xycoords="axes fraction", fontsize=8,
             bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", ec="gray"))

# ── Panel 2: Residuals vs Fitted (Heteroskedasticity check) ──────────────────
ax2 = fig.add_subplot(gs[0, 1])
idx_sample = np.random.choice(len(fitted_arr), size=min(8000, len(fitted_arr)),
                               replace=False)
ax2.scatter(fitted_arr[idx_sample], resid_arr[idx_sample],
            alpha=0.2, s=4, color="#70AD47", rasterized=True)
ax2.axhline(0, color="red", lw=1.5, linestyle="--")
# Lowess smoother
from statsmodels.nonparametric.smoothers_lowess import lowess
lw_fit = lowess(resid_arr[idx_sample], fitted_arr[idx_sample], frac=0.2)
ax2.plot(lw_fit[:, 0], lw_fit[:, 1], color="navy", lw=1.8,
         label="Lowess smoother")
ax2.set_title("Residuals vs Fitted\n(Heteroskedasticity check)", fontsize=11)
ax2.set_xlabel("Fitted Values")
ax2.set_ylabel("Residuals")
ax2.legend(fontsize=8)

# ── Panel 3: Scale-Location plot ──────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
sqrt_abs_resid = np.sqrt(np.abs(resid_arr))
ax3.scatter(fitted_arr[idx_sample], sqrt_abs_resid[idx_sample],
            alpha=0.2, s=4, color="#ED7D31", rasterized=True)
lw_sl = lowess(sqrt_abs_resid[idx_sample], fitted_arr[idx_sample], frac=0.2)
ax3.plot(lw_sl[:, 0], lw_sl[:, 1], color="navy", lw=1.8)
ax3.set_title("Scale-Location\n(Homoskedasticity check)", fontsize=11)
ax3.set_xlabel("Fitted Values")
ax3.set_ylabel("√|Residuals|")

# ── Panel 4: QQ plot of product random effects ────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
# re_prod stored in df_mod via the null product model in section 4
re_prod_vals = df_mod["re_prod"].dropna().unique()
re_prod_vals = re_prod_vals[~np.isnan(re_prod_vals)]
(osm4, osr4), (s4, i4, _) = stats.probplot(re_prod_vals, dist="norm")
ax4.scatter(osm4, osr4, alpha=0.7, s=25, color="#9966CC")
x4 = np.array([osm4.min(), osm4.max()])
ax4.plot(x4, s4 * x4 + i4, "r-", lw=1.5)
ax4.set_title("QQ Plot — Product Random Effects\n"
              f"(n={len(re_prod_vals)} products)", fontsize=11)
ax4.set_xlabel("Theoretical Quantiles")
ax4.set_ylabel("Random Intercepts")

# ── Panel 5: QQ plot of brand random effects ──────────────────────────────────
ax5 = fig.add_subplot(gs[2, 0])
(osm5, osr5), (s5, i5, _) = stats.probplot(re_brand, dist="norm")
ax5.scatter(osm5, osr5, alpha=0.7, s=25, color="#FF6B6B")
x5 = np.array([osm5.min(), osm5.max()])
ax5.plot(x5, s5 * x5 + i5, "r-", lw=1.5)
ax5.set_title("QQ Plot — Brand Random Effects\n"
              f"(n={len(re_brand)} brands)", fontsize=11)
ax5.set_xlabel("Theoretical Quantiles")
ax5.set_ylabel("Random Intercepts")

# ── Panel 6: Histogram of residuals ──────────────────────────────────────────
ax6 = fig.add_subplot(gs[2, 1])
ax6.hist(resid_arr, bins=80, color="#4472C4", alpha=0.7, density=True,
         edgecolor="none")
xmin, xmax = ax6.get_xlim()
x_norm = np.linspace(xmin, xmax, 300)
ax6.plot(x_norm,
         stats.norm.pdf(x_norm, resid_arr.mean(), resid_arr.std()),
         "r-", lw=2, label="Normal fit")
ax6.set_title("Residual Distribution", fontsize=11)
ax6.set_xlabel("Residual")
ax6.set_ylabel("Density")
ax6.legend(fontsize=8)

diag_path = os.path.join(OUT, "assumption_diagnostics.png")
fig.savefig(diag_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"  Saved → {diag_path}")

  Saved → /content/sample_data/assumption_diagnostics.png


In [34]:
# =============================================================================
#  6. MULTICOLLINEARITY — VIF  (mirrors check_collinearity())
# =============================================================================
print("\n[6/8]  Multicollinearity check (VIF)...")

vif_formula = (
    "distortion ~ VP1 + IMG1 + VP_IMG + "
    "sent_VP + sent_IMG + sent_VP_IMG + "
    "prior_mean_rating + log_length + log_helpful + log_reviews + SUBJ"
)
vif_ols  = smf.ols(vif_formula, data=df_mod).fit()
X_vif    = vif_ols.model.exog
vif_names = vif_ols.model.exog_names

vif_df = pd.DataFrame({
    "Variable": vif_names[1:],          # skip intercept
    "VIF":      [variance_inflation_factor(X_vif, i + 1)
                 for i in range(len(vif_names) - 1)],
}).round(2)

vif_labels = {
    "VP1":              "VP",
    "IMG1":             "IMG",
    "VP_IMG":           "VP × IMG",
    "sent_VP":          "Sentiment × VP",
    "sent_IMG":         "Sentiment × IMG",
    "sent_VP_IMG":      "Sentiment × VP × IMG",
    "prior_mean_rating":"Prior Mean Rating",
    "log_length":       "Log(Review Length)",
    "log_helpful":      "Log(Helpful Votes+1)",
    "log_reviews":      "Log(Total Reviews)",
    "SUBJ":             "Subjectivity",
}
vif_df["Label"] = vif_df["Variable"].map(vif_labels)
vif_df["Flag"]  = vif_df["VIF"].apply(
    lambda v: "⚠ SEVERE"   if v > 10 else
              "△ MODERATE" if v > 5  else
              "✓ OK"
)

print("\n  ── VIF Table ──")
print(vif_df[["Label", "VIF", "Flag"]].to_string(index=False))

# ── VIF bar chart ─────────────────────────────────────────────────────────────
fig_v, ax_v = plt.subplots(figsize=(9, 5))
colors = ["#C00000" if v > 10 else "#ED7D31" if v > 5 else "#70AD47"
          for v in vif_df["VIF"]]
bars = ax_v.barh(vif_df["Label"], vif_df["VIF"], color=colors, edgecolor="white")
ax_v.axvline(5,  color="orange", linestyle="--", lw=1.2, label="Moderate (VIF=5)")
ax_v.axvline(10, color="red",    linestyle="--", lw=1.2, label="Severe (VIF=10)")
for bar, val in zip(bars, vif_df["VIF"]):
    ax_v.text(val + 0.1, bar.get_y() + bar.get_height() / 2,
              f"{val:.1f}", va="center", fontsize=8)
ax_v.set_xlabel("VIF")
ax_v.set_title("Variance Inflation Factors — Interaction Terms", fontweight="bold")
ax_v.legend(fontsize=8)
ax_v.invert_yaxis()
plt.tight_layout()
vif_path = os.path.join(OUT, "vif_plot.png")
fig_v.savefig(vif_path, dpi=150, bbox_inches="tight")
plt.close(fig_v)
print(f"  Saved → {vif_path}")


[6/8]  Multicollinearity check (VIF)...

  ── VIF Table ──
               Label   VIF       Flag
                  VP  1.23       ✓ OK
                 IMG 10.28   ⚠ SEVERE
            VP × IMG  9.87 △ MODERATE
      Sentiment × VP  1.27       ✓ OK
     Sentiment × IMG  9.89 △ MODERATE
Sentiment × VP × IMG  9.56 △ MODERATE
   Prior Mean Rating  1.13       ✓ OK
  Log(Review Length)  1.21       ✓ OK
Log(Helpful Votes+1)  1.20       ✓ OK
  Log(Total Reviews)  1.10       ✓ OK
        Subjectivity  1.16       ✓ OK
  Saved → /content/sample_data/vif_plot.png


In [23]:
# =============================================================================
#  7. OUTLIER DETECTION  (mirrors check_outliers())
#     Uses Cook's Distance from the OLS auxiliary model
# =============================================================================
print("\n[7/8]  Outlier / influence detection (Cook's Distance)...")

cook_ols   = smf.ols(vif_formula, data=df_mod).fit()
influence  = cook_ols.get_influence()
cooks_d    = influence.cooks_distance[0]
cook_thresh = 4 / len(df_mod)
n_outliers  = (cooks_d > cook_thresh).sum()

print(f"  Cook's D threshold = 4/n = {cook_thresh:.6f}")
print(f"  Observations exceeding threshold: {n_outliers} "
      f"({100*n_outliers/len(df_mod):.2f}%)")

fig_c, ax_c = plt.subplots(figsize=(10, 4))
ax_c.scatter(range(len(cooks_d)), cooks_d,
             s=3, alpha=0.4, color="#4472C4", rasterized=True)
ax_c.axhline(cook_thresh, color="red", lw=1.5, linestyle="--",
             label=f"Threshold = 4/n = {cook_thresh:.5f}")
ax_c.set_xlabel("Observation Index")
ax_c.set_ylabel("Cook's Distance")
ax_c.set_title("Cook's Distance — Influence Diagnostics", fontweight="bold")
ax_c.legend(fontsize=9)
plt.tight_layout()
cook_path = os.path.join(OUT, "cooks_distance.png")
fig_c.savefig(cook_path, dpi=150, bbox_inches="tight")
plt.close(fig_c)
print(f"  Saved → {cook_path}")


[7/8]  Outlier / influence detection (Cook's Distance)...
  Cook's D threshold = 4/n = 0.000072
  Observations exceeding threshold: 2731 (4.91%)
  Saved → /content/sample_data/cooks_distance.png


In [35]:
# =============================================================================
#  8. ROBUST STANDARD ERRORS + FULL RESULTS TABLE
#     Cluster-robust SEs (clustered at product level) and HC3 for comparison.
#     OLS with product + brand dummies partialling out group-level means,
#     matching the mixed-model fixed-effect estimates.
# =============================================================================
print("\n[8/8]  Computing robust standard errors...")

formula_full = (
    "distortion ~ VP1 + IMG1 + VP_IMG + "
    "sent_VP + sent_IMG + sent_VP_IMG + "
    "prior_mean_rating + log_length + log_helpful + log_reviews + SUBJ + "
    "C(quarter_str) + C(product_str) + C(brand_str)"
)

# ── OLS (plain) ───────────────────────────────────────────────────────────────
ols_plain = smf.ols(formula_full, data=df_mod).fit()

# ── Cluster-robust SEs (clustered at product) ─────────────────────────────────
cluster_grp   = df_mod.loc[ols_plain.model.data.row_labels, "product_str"].values
robust_clust  = ols_plain.get_robustcov_results(cov_type="cluster",
                                                 groups=cluster_grp)

# ── HC3 heteroskedasticity-robust (no clustering) ────────────────────────────
robust_hc3    = ols_plain.get_robustcov_results(cov_type="HC3")

# ── Extract focal coefficients only ──────────────────────────────────────────
focal_terms = [
    "Intercept",
    "VP1", "IMG1", "VP_IMG",
    "sent_VP", "sent_IMG", "sent_VP_IMG",
    "prior_mean_rating", "log_length",
    "log_helpful", "log_reviews", "SUBJ",
]

def _extract(result, terms):
    t = result.summary2().tables[1]
    return t.loc[terms, ["Coef.", "Std.Err.", "t", "P>|t|", "[0.025", "0.975]"]]

tab_clust = _extract(robust_clust, focal_terms)
tab_hc3   = _extract(robust_hc3,   focal_terms)
tab_ols   = _extract(ols_plain,    focal_terms)

# ── Significance stars helper ─────────────────────────────────────────────────
def _stars(p):
    if   p < 0.001: return "***"
    elif p < 0.01:  return "**"
    elif p < 0.05:  return "*"
    elif p < 0.10:  return "†"
    return ""

# ── Build combined output table ───────────────────────────────────────────────
var_labels = {
    "Intercept":         "Intercept",
    "VP1":               "VP (Verified Purchase)",
    "IMG1":              "IMG (Image Present)",
    "VP_IMG":            "VP × IMG",
    "sent_VP":           "Sentiment × VP          [H1]",
    "sent_IMG":          "Sentiment × IMG         [H1]",
    "sent_VP_IMG":       "Sentiment × VP × IMG    [H1]",
    "prior_mean_rating": "Prior Mean Rating",
    "log_length":        "Log(Review Length)",
    "log_helpful":       "Log(Helpful Votes+1)",
    "log_reviews":       "Log(Total Reviews)",
    "SUBJ":              "Subjectivity",
}

rows = []
for term in focal_terms:
    coef  = tab_clust.loc[term, "Coef."]
    se_c  = tab_clust.loc[term, "Std.Err."]
    t_c   = tab_clust.loc[term, "t"]
    p_c   = tab_clust.loc[term, "P>|t|"]
    ci_lo = tab_clust.loc[term, "[0.025"]
    ci_hi = tab_clust.loc[term, "0.975]"]
    se_hc3 = tab_hc3.loc[term, "Std.Err."]
    se_ols = tab_ols.loc[term, "Std.Err."]
    vif_val = vif_df.loc[vif_df["Variable"] == term, "VIF"].values
    vif_val = vif_val[0] if len(vif_val) else np.nan
    rows.append({
        "Variable":       var_labels.get(term, term),
        "Coef.":          round(coef, 4),
        "SE (Cluster)":   round(se_c, 4),
        "SE (HC3)":       round(se_hc3, 4),
        "SE (OLS naive)": round(se_ols, 4),
        "t":              round(t_c, 3),
        "p-value":        round(p_c, 4),
        "Sig.":           _stars(p_c),
        "CI 2.5%":        round(ci_lo, 4),
        "CI 97.5%":       round(ci_hi, 4),
        "VIF":            round(vif_val, 2) if not np.isnan(vif_val) else "—",
    })

results_df = pd.DataFrame(rows)

print("\n  ── Full Results Table (Cluster-Robust SEs) ──")
print(f"  N={len(df_mod):,} | Products={df_mod['product_str'].nunique()} | "
      f"Brands={df_mod['brand_str'].nunique()} | "
      f"Quarters={df_mod['quarter_str'].nunique()}")
print(results_df[
    ["Variable", "Coef.", "SE (Cluster)", "t", "p-value", "Sig.", "VIF"]
].to_string(index=False))
print("  Sig. codes: *** p<0.001  ** p<0.01  * p<0.05  † p<0.10")

# ── SE inflation ratio (Cluster vs naive OLS) ─────────────────────────────────
print("\n  ── SE Inflation (Cluster / Naive OLS) ──")
for _, row in results_df.iterrows():
    if isinstance(row["VIF"], float):
        ratio = row["SE (Cluster)"] / row["SE (OLS naive)"]
        print(f"  {row['Variable']:<40} ratio = {ratio:.2f}x")




[8/8]  Computing robust standard errors...


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 446, but rank is 31
  warnings.warn('covariance of constraints does not have full '



  ── Full Results Table (Cluster-Robust SEs) ──
  N=55,668 | Products=322 | Brands=95 | Quarters=21
                    Variable    Coef.  SE (Cluster)       t  p-value Sig.    VIF
                   Intercept -59.7087        4.1419 -14.416   0.0000  ***      —
      VP (Verified Purchase)   0.2656        0.0520   5.109   0.0000  ***   1.23
         IMG (Image Present)   0.3254        0.0664   4.903   0.0000  ***  10.28
                    VP × IMG  -0.3785        0.0842  -4.494   0.0000  ***   9.87
Sentiment × VP          [H1]  -0.0666        0.0109  -6.116   0.0000  ***   1.27
Sentiment × IMG         [H1]  -0.0020        0.0714  -0.027   0.9782        9.89
Sentiment × VP × IMG    [H1]   0.0340        0.0704   0.483   0.6293        9.56
           Prior Mean Rating   0.0736        0.0407   1.808   0.0716    †   1.13
          Log(Review Length)  -0.4504        0.0171 -26.278   0.0000  ***   1.21
        Log(Helpful Votes+1)  -0.1495        0.0317  -4.710   0.0000  ***    1.2
        

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 446, but rank is 353
  warnings.warn('covariance of constraints does not have full '


In [36]:
# =============================================================================
#  9. VARIANCE DECOMPOSITION — ICC  (H2: Structured Distortion)
#     performance::icc() equivalent
#     Null models at each level provide unconditional ICCs
# =============================================================================
print("\n── H2: Variance Decomposition (ICC) ─────────────────────────────────")

# All three variance components already estimated above:
#   var_product  → from null product model  (section 4)
#   var_brand_re → from mlm_fit.vcomp[0]    (section 4)
#   var_resid_re → from mlm_fit.scale       (section 4)
var_prod  = var_product
var_brand = var_brand_re
var_resid = var_resid_re
var_total = var_prod + var_brand + var_resid

icc_prod  = var_prod  / var_total
icc_brand = var_brand / var_total
icc_resid = var_resid / var_total

print(f"\n  Variance components (null models):")
print(f"    Var(product)  = {var_prod:.4f}  | ICC = {icc_prod:.4f} "
      f"({100*icc_prod:.1f}%)")
print(f"    Var(brand)    = {var_brand:.4f}  | ICC = {icc_brand:.4f} "
      f"({100*icc_brand:.1f}%)")
print(f"    Var(residual) = {var_resid:.4f}  | ICC = {icc_resid:.4f} "
      f"({100*icc_resid:.1f}%)")
print(f"    Var(total)    = {var_total:.4f}")
print(f"\n  → H2 support: {(icc_prod + icc_brand)*100:.1f}% of distortion "
      f"variance is attributable to product + brand levels.")

# ── ICC pie chart ─────────────────────────────────────────────────────────────
fig_icc, axes_icc = plt.subplots(1, 2, figsize=(12, 5))
fig_icc.suptitle("H2 — Variance Decomposition of Distortion",
                  fontweight="bold", fontsize=13)

# Pie
labels_pie  = ["Product\n(between-product)",
               "Brand\n(between-brand)",
               "Residual\n(within)"]
sizes_pie   = [icc_prod, icc_brand, icc_resid]
colors_pie  = ["#4472C4", "#ED7D31", "#A9A9A9"]
explode     = (0.05, 0.05, 0)

axes_icc[0].pie(sizes_pie, labels=labels_pie, autopct="%1.1f%%",
                colors=colors_pie, explode=explode,
                textprops={"fontsize": 10}, startangle=90)
axes_icc[0].set_title("ICC Partition", fontsize=11)

# Bar with variance magnitudes
var_names = ["Product", "Brand", "Residual"]
var_vals  = [var_prod, var_brand, var_resid]
b_bars    = axes_icc[1].bar(var_names, var_vals, color=colors_pie,
                             edgecolor="white", width=0.5)
for bar, val in zip(b_bars, var_vals):
    axes_icc[1].text(bar.get_x() + bar.get_width() / 2, val + 0.003,
                     f"{val:.4f}", ha="center", fontsize=9)
axes_icc[1].set_ylabel("Variance")
axes_icc[1].set_title("Variance Components (σ²)", fontsize=11)
axes_icc[1].set_ylim(0, max(var_vals) * 1.15)

plt.tight_layout()
icc_path = os.path.join(OUT, "icc_variance_decomposition.png")
fig_icc.savefig(icc_path, dpi=150, bbox_inches="tight")
plt.close(fig_icc)
print(f"  Saved → {icc_path}")


── H2: Variance Decomposition (ICC) ─────────────────────────────────

  Variance components (null models):
    Var(product)  = 0.0565  | ICC = 0.0363 (3.6%)
    Var(brand)    = 0.0321  | ICC = 0.0206 (2.1%)
    Var(residual) = 1.4681  | ICC = 0.9430 (94.3%)
    Var(total)    = 1.5568

  → H2 support: 5.7% of distortion variance is attributable to product + brand levels.
  Saved → /content/sample_data/icc_variance_decomposition.png


In [37]:
# =============================================================================
#  10. SAVE TABLES TO CSV
# =============================================================================
results_df.to_csv(os.path.join(OUT, "H1_H2_robust_results.csv"), index=False)
vif_df[["Label","VIF","Flag"]].to_csv(os.path.join(OUT, "vif_table.csv"), index=False)

icc_out = pd.DataFrame({
    "Level":      ["Product", "Brand", "Residual", "Total"],
    "Variance":   [var_prod,  var_brand, var_resid, var_total],
    "ICC":        [icc_prod,  icc_brand, icc_resid, 1.0],
    "ICC_pct":    [f"{100*icc_prod:.1f}%", f"{100*icc_brand:.1f}%",
                   f"{100*icc_resid:.1f}%", "100.0%"],
}).round(4)
icc_out.to_csv(os.path.join(OUT, "H2_icc_variance.csv"), index=False)

print(f"\n  Saved → H1_H2_robust_results.csv")
print(f"  Saved → vif_table.csv")
print(f"  Saved → H2_icc_variance.csv")


  Saved → H1_H2_robust_results.csv
  Saved → vif_table.csv
  Saved → H2_icc_variance.csv


In [38]:

# =============================================================================
#  11. COMBINED SUMMARY PLOT  (H1 coefficient plot)
# =============================================================================
h1_terms = ["sent_VP", "sent_IMG", "sent_VP_IMG"]
h1_labels = [
    "Sentiment × VP\n[H1: VP amplification]",
    "Sentiment × IMG\n[H1: IMG amplification]",
    "Sentiment × VP × IMG\n[H1: joint amplification]",
]
h1_coefs  = [tab_clust.loc[t, "Coef."]   for t in h1_terms]
h1_ci_lo  = [tab_clust.loc[t, "[0.025"]  for t in h1_terms]
h1_ci_hi  = [tab_clust.loc[t, "0.975]"]  for t in h1_terms]
h1_pvals  = [tab_clust.loc[t, "P>|t|"]   for t in h1_terms]

fig_h1, ax_h1 = plt.subplots(figsize=(8, 4))
y_pos = np.arange(len(h1_terms))
ax_h1.errorbar(
    h1_coefs, y_pos,
    xerr=[np.array(h1_coefs) - np.array(h1_ci_lo),
          np.array(h1_ci_hi) - np.array(h1_coefs)],
    fmt="o", color="#4472C4", ecolor="#4472C4",
    elinewidth=2, capsize=5, markersize=8,
)
ax_h1.axvline(0, color="red", linestyle="--", lw=1.5, label="Null effect")
for i, (coef, p) in enumerate(zip(h1_coefs, h1_pvals)):
    sig = _stars(p)
    ax_h1.text(coef + 0.005, y_pos[i] + 0.06,
               f"β={coef:.3f}{sig}", fontsize=9)
ax_h1.set_yticks(y_pos)
ax_h1.set_yticklabels(h1_labels, fontsize=10)
ax_h1.set_xlabel("Coefficient (Cluster-Robust 95% CI)", fontsize=10)
ax_h1.set_title("H1 — Platform Cue × Sentiment Amplification Effects",
                fontweight="bold", fontsize=11)
ax_h1.legend(fontsize=9)
plt.tight_layout()
h1_path = os.path.join(OUT, "H1_coefficient_plot.png")
fig_h1.savefig(h1_path, dpi=150, bbox_inches="tight")
plt.close(fig_h1)
print(f"  Saved → {h1_path}")

print("\n" + "=" * 70)
print("  ALL DONE.")
print(f"  Outputs written to: {OUT}")
print("=" * 70)

  Saved → /content/sample_data/H1_coefficient_plot.png

  ALL DONE.
  Outputs written to: /content/sample_data


In [42]:
# =============================================================================
#  H3 — HETEROGENEOUS AMPLIFICATION ACROSS PRODUCTS
#  Tests whether the Sentiment × VP and Sentiment × IMG amplification effects
#  vary significantly across products via random slopes.
#
#  Mirrors R:
#    model_base  → (1 | ASIN) + (1 | brand) + (1 | quarter)
#    model_h3_v2 → (0 + sentiment_c:VP1  | ASIN)
#                  (0 + sentiment_c:IMG_num | ASIN)
#    anova(model_base_ml, model_h3_v2_ml)
#
#  Python strategy:
#    Random slopes via vc_formula (scalar variance per product) — NOT
#    re_formula, which estimates a full 2×2 covariance matrix and becomes
#    singular when slope variance ≈ 0 (sent_IMG case).
#    vc_formula matches R's (0 + ...) syntax: slope variance only,
#    no correlated random intercept term.
#    All models fitted with ML (reml=False) so LRT is valid.
# =============================================================================
print("\n" + "=" * 70)
print("  H3 — Heterogeneous Amplification Across Products")
print("=" * 70)

# ── H3 dataset ────────────────────────────────────────────────────────────────
# Keep products with >= 5 reviews AND within-product variation in both
# VP and IMG — mirrors R's filter(n()>=5) + filter(var(IMG_num)>0)
df_h3 = df_mod.copy()
df_h3 = df_h3.groupby('product_str').filter(lambda x: len(x) >= 5)
df_h3 = df_h3.groupby('product_str').filter(lambda x: x['VP1'].var() > 0)
df_h3 = df_h3.groupby('product_str').filter(lambda x: x['IMG1'].var() > 0)
df_h3 = df_h3.reset_index(drop=True)

print(f"\n  H3 dataset : N={len(df_h3):,}  "
      f"products={df_h3['product_str'].nunique()}  "
      f"brands={df_h3['brand_str'].nunique()}")
print(f"  Mean reviews/product : "
      f"{len(df_h3)/df_h3['product_str'].nunique():.1f}")

# ── Shared formula ─────────────────────────────────────────────────────────────
# sentiment_c main effect excluded — orthogonal to distortion by Stage-1
# construction (confirmed: Pearson r ≈ 0, F-test p = 0.50)
h3_formula = (
    "distortion ~ VP1 + IMG1 + VP_IMG + "
    "sent_VP + sent_IMG + sent_VP_IMG + "
    "prior_mean_rating + log_length + log_helpful + log_reviews + SUBJ + "
    "C(quarter_str)"
)

# ── Model factory ─────────────────────────────────────────────────────────────
# extra_vc : dict of additional vc_formula entries for random slopes
# brand RE always included as the base vc entry
def _fit_h3(extra_vc=None):
    vc = {"brand": "0 + C(brand_str)"}          # brand random intercept
    if extra_vc:
        vc.update(extra_vc)
    return smf.mixedlm(
        h3_formula, df_h3,
        groups     = df_h3['product_str'],        # product random intercept
        vc_formula = vc
    ).fit(reml=False, method='powell', maxiter=500)

# ── Fit four models ────────────────────────────────────────────────────────────
print("\n  Fitting models (ML for LRT validity)...")
m0_h3 = _fit_h3()                                         # intercepts only
m1_h3 = _fit_h3({"vp_slope":  "0 + sent_VP"})            # + VP  slope
m2_h3 = _fit_h3({"img_slope": "0 + sent_IMG"})           # + IMG slope
m3_h3 = _fit_h3({"vp_slope":  "0 + sent_VP",             # + both slopes
                  "img_slope": "0 + sent_IMG"})

for name, m in [("M0 baseline",          m0_h3),
                ("M1 +slope sent_VP",     m1_h3),
                ("M2 +slope sent_IMG",    m2_h3),
                ("M3 +both slopes",       m3_h3)]:
    print(f"  {name:<25} converged={m.converged}  logLik={m.llf:.4f}")

# ── LRT helper ────────────────────────────────────────────────────────────────
def _lrt(m_base, m_ext, df_diff):
    chi2 = 2 * (m_ext.llf - m_base.llf)
    p    = stats.chi2.sf(chi2, df_diff)
    sig  = ('***' if p < 0.001 else '**' if p < 0.01
            else '*' if p < 0.05 else 'n.s.')
    return chi2, p, sig

# ── LRT table ─────────────────────────────────────────────────────────────────
print("\n  ── Likelihood Ratio Tests (all vs M0 baseline) ──")
print(f"  {'Comparison':<42} {'χ²':>8}  {'df':>3}  {'p-value':>12}  Sig.")
print("  " + "-" * 72)

lrt_specs = [
    ("M0 vs M1  (+random slope Sent×VP)",  m1_h3, 2),
    ("M0 vs M2  (+random slope Sent×IMG)", m2_h3, 2),
    ("M0 vs M3  (+both random slopes)",    m3_h3, 4),
]
lrt_records = []
for label, m_ext, ddf in lrt_specs:
    chi2, p, sig = _lrt(m0_h3, m_ext, ddf)
    print(f"  {label:<42} {chi2:>8.3f}  {ddf:>3}  {p:>12.4e}  {sig}")
    lrt_records.append({"Comparison": label, "chi2": round(chi2, 3),
                         "df": ddf, "p_value": round(p, 6), "Sig.": sig})

# ── Variance components ────────────────────────────────────────────────────────
# vcomp index: [0]=brand, [1]=added slope (M1/M2); M3: [0]=brand,[1]=VP,[2]=IMG
var_vp  = float(m1_h3.vcomp[1])
var_img = float(m2_h3.vcomp[1])

print(f"\n  ── Random Slope Variances (σ² per product) ──")
print(f"  σ²(Sent×VP  across products) = {var_vp:.6f}  "
      f"σ = {np.sqrt(var_vp):.4f}")
print(f"  σ²(Sent×IMG across products) = {var_img:.6f}  "
      f"σ = {np.sqrt(var_img):.4f}")

# ── 95% plausible range of product-specific slopes ────────────────────────────
fe_vp  = float(m1_h3.fe_params.get('sent_VP',  0))
fe_img = float(m2_h3.fe_params.get('sent_IMG', 0))
sd_vp  = np.sqrt(var_vp)
sd_img = np.sqrt(var_img)

print(f"\n  ── Plausible Range of Product-Specific Slopes (mean ± 1.96 SD) ──")
print(f"  Sent×VP  : fixed β={fe_vp:.4f}  "
      f"range=[{fe_vp-1.96*sd_vp:.4f}, {fe_vp+1.96*sd_vp:.4f}]")
print(f"  Sent×IMG : fixed β={fe_img:.4f}  "
      f"range=[{fe_img-1.96*sd_img:.4f}, {fe_img+1.96*sd_img:.4f}]")

# ── H3 visualisation: forest-style plot of plausible slope distributions ───────
fig_h3, axes = plt.subplots(1, 2, figsize=(12, 5))
fig_h3.suptitle(
    "H3 — Cross-Product Heterogeneity in Amplification Effects",
    fontweight="bold", fontsize=13
)

for ax, fe, sd, var, label, color in [
    (axes[0], fe_vp,  sd_vp,  var_vp,
     "Sent × VP\n(Verified Purchase)", "#4472C4"),
    (axes[1], fe_img, sd_img, var_img,
     "Sent × IMG\n(Image Present)",    "#ED7D31"),
]:
    x = np.linspace(fe - 4*sd, fe + 4*sd, 400)
    y = stats.norm.pdf(x, fe, sd)
    ax.plot(x, y, color=color, lw=2.5)
    ax.fill_between(x, y,
                    where=(x >= fe - 1.96*sd) & (x <= fe + 1.96*sd),
                    alpha=0.25, color=color, label="95% plausible range")
    ax.axvline(fe,  color=color,  lw=2,   linestyle='-',  label=f"Mean β={fe:.3f}")
    ax.axvline(0,   color='red',  lw=1.5, linestyle='--', label="Null (β=0)")
    ax.set_xlabel("Product-specific slope")
    ax.set_ylabel("Density")
    ax.set_title(f"{label}\nσ²={var:.5f},  σ={sd:.4f}", fontsize=11)
    ax.legend(fontsize=8)

plt.tight_layout()
h3_plot_path = os.path.join(OUT, "H3_slope_distributions.png")
fig_h3.savefig(h3_plot_path, dpi=150, bbox_inches="tight")
plt.close(fig_h3)
print(f"\n  Saved → {h3_plot_path}")

# ── Save LRT table ─────────────────────────────────────────────────────────────
h3_df = pd.DataFrame(lrt_records)
h3_df.to_csv(os.path.join(OUT, "H3_lrt_results.csv"), index=False)
print(f"  Saved → H3_lrt_results.csv")

print(f"""
  ── Summary ──
  H3 is supported for both platform cues:
    Sent×VP  : χ²(2)={lrt_records[0]['chi2']:.3f}, p={lrt_records[0]['p_value']:.2e} {lrt_records[0]['Sig.']}
    Sent×IMG : χ²(2)={lrt_records[1]['chi2']:.3f}, p={lrt_records[1]['p_value']:.2e} {lrt_records[1]['Sig.']}
  The VP amplification effect varies more consistently across products
  (σ={sd_vp:.4f}) than the IMG effect (σ={sd_img:.4f}), though both
  are statistically significant. LRT for both slopes jointly confirms
  the overall heterogeneity structure.
""")


  H3 — Heterogeneous Amplification Across Products

  H3 dataset : N=53,565  products=132  brands=56
  Mean reviews/product : 405.8

  Fitting models (ML for LRT validity)...


/usr/local/lib/python3.12/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/usr/local/lib/python3.12/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


  M0 baseline               converged=True  logLik=-86301.7264
  M1 +slope sent_VP         converged=True  logLik=-86281.2969
  M2 +slope sent_IMG        converged=True  logLik=-86297.2547
  M3 +both slopes           converged=True  logLik=-86279.6263

  ── Likelihood Ratio Tests (all vs M0 baseline) ──
  Comparison                                       χ²   df       p-value  Sig.
  ------------------------------------------------------------------------
  M0 vs M1  (+random slope Sent×VP)            40.859    2    1.3415e-09  ***
  M0 vs M2  (+random slope Sent×IMG)            8.943    2    1.1428e-02  *
  M0 vs M3  (+both random slopes)              44.200    4    5.8299e-09  ***

  ── Random Slope Variances (σ² per product) ──
  σ²(Sent×VP  across products) = 0.003646  σ = 0.0604
  σ²(Sent×IMG across products) = 0.011260  σ = 0.1061

  ── Plausible Range of Product-Specific Slopes (mean ± 1.96 SD) ──
  Sent×VP  : fixed β=-0.0529  range=[-0.1712, 0.0655]
  Sent×IMG : fixed β=0.0228  